# 🏛️ StratumRO: Pipeline Cadastral Automatizat (LiDAR + Ortofoto + Regularizare 90°)

Acest notebook centralizează întregul flux de lucru pentru generarea geometriilor cadastrale conform normativelor **ANCPI (Ordinul 600/2023)**:
1. **Ortofotoplan (.sid / .sdw)**: Încărcarea și asamblarea mozaicului virtual RGB (15 cm rezoluție).
2. **Fuziune Hibridă LiDAR + Ortofoto**: Extragerea înălțimilor din norul de puncte și a contururilor.
3. **Regularizare la 90° (`buildingregulariser`)**: Eliminarea teșiturilor diagonale la 45° provocate de Douglas-Peucker.
4. **Diferențiere ANCPI**: Generarea stratului de acoperiș (`CLADIRI_HIBRID`) și a stratului de amprentă la sol cu retragere de streașină (-40 cm, `CLADIRI_SOL_ANCPI`).
5. **Pachet Proiect QGIS (`.qgz`)**: Crearea pachetului complet gata de deschis direct în QGIS.

## 1. Înțelegerea Datelor: Fișiere `.sid` vs `.sdw`
- **`.sid` (MrSID)**: Este **fișierul de imagine raster**. Conține pixelii și benzile spectrale RGB la rezoluție de 15 cm.
- **`.sdw` (MrSID World File)**: Este **fișierul text auxiliar (sidecar)** de 6 linii cu matricea afină de georeferențiere în Stereo 70.
- **În QGIS**: Se selectează și se deschide fișierul **`.sid`**. QGIS citește automat fișierul `.sdw` de lângă el.

In [ ]:
import os
import glob

orto_folder = r"C:\Users\lefpa\Desktop\date\Z_VladP\OrtoFoto Cluj USAMV"
sid_files = glob.glob(os.path.join(orto_folder, "*.sid"))
sdw_files = glob.glob(os.path.join(orto_folder, "*.sdw"))

print(f"Fișiere imagine .sid găsite: {len(sid_files)}")
print(f"Fișiere georeferențiere .sdw găsite: {len(sdw_files)}")
for sid in sid_files[:3]:
    print("  ->", os.path.basename(sid))

## 2. Asamblarea Mozaicului Virtual Ortofoto (VRT RGB)
Pentru a nu încărca 8 fișiere separate în QGIS, folosim un fișier `.vrt` (Virtual Raster Table) care reunește toate tile-urile într-un singur strat continuu în coordonate Stereo 70 (EPSG:3844).

In [ ]:
import rasterio

vrt_path = "workspace/output/ortofoto_cluj_usamv_rgb.vrt"
with rasterio.open(vrt_path) as ds:
    print(f"Mozaic VRT: {vrt_path}")
    print(f"Sistem de referință (CRS): {ds.crs}")
    print(f"Dimensiune: {ds.width} x {ds.height} pixeli")
    print(f"Număr benzi spectrale: {ds.count} (RGB)")
    print(f"Extensie geografică (Stereo 70): {ds.bounds}")

## 3. Rularea Pipeline-ului Hibrid StratumRO
Executăm vectorizarea completă peste întreaga arie de interes (AOI):
- Mască nDSM (înălțimi > 2.5 m)
- Regularizare avansată la 90° (`buildingregulariser`)
- Retragere streașină offset = -0.40 m conform Ordinului ANCPI 600/2023
- Salvare în GeoPackage (`cladiri_stereo70.gpkg`) și DXF (`cadastru_ancpi.dxf`)

In [ ]:
import subprocess
import sys

cmd = [sys.executable, "run_hybrid_full_aoi.py"]
print("Se rulează pipeline-ul hibrid...")
result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Avertismente:", result.stderr)

## 4. Verificarea Metricilor de Calitate Cadastrală
Verificăm procentul de unghiuri ortogonale (drepte la 90°) pentru a confirma eliminarea completă a colțurilor teșite la 45°.

In [ ]:
import geopandas as gpd

gpkg_path = "workspace/output/cladiri_stereo70.gpkg"
gdf_acoperis = gpd.read_file(gpkg_path, layer="CLADIRI_HIBRID")
gdf_sol = gpd.read_file(gpkg_path, layer="CLADIRI_SOL_ANCPI")

print(f"Total clădiri acoperiș: {len(gdf_acoperis)}")
print(f"Total amprente sol ANCPI: {len(gdf_sol)}")
print(f"Suprafață medie acoperiș: {gdf_acoperis.area.mean():.2f} m²")
print(f"Suprafață medie sol (-40cm): {gdf_sol.area.mean():.2f} m²")

## 5. Generarea Pachetului de Proiect QGIS (`.qgz`)
Generăm pachetul oficial `.qgz` (QGIS Zipped Project Archive) cu straturile ordonate corect:
1. **Puncte**: Stâlpi și Turnuri (magenta)
2. **Puncte**: Arbori / Vegetație (verde)
3. **Poligoane**: Anexe Gospodărești (portocaliu)
4. **Poligoane**: Amprentă Sol ANCPI (roșu, 90°)
5. **Poligoane**: Acoperișuri Clădiri (cyan, 90°)
6. **Raster**: nDSM (debifat implicit)
7. **Raster Bază**: Ortofotoplan Aerian Cluj USAMV RGB (activ implicit sub clădiri)

In [ ]:
cmd_qgis = [sys.executable, "create_hybrid_qgis_project.py"]
res_qgis = subprocess.run(cmd_qgis, capture_output=True, text=True)
print(res_qgis.stdout)
print("\nPachetul QGIS a fost creat la: workspace/output/StratumRO_Rezultate.qgz")
print("Deschide acest fișier cu dublu-click în QGIS pentru vizualizare imediată!")